In [5]:
import pandas as pd
import numpy as np
import re

# ============================================================
# KONFIGURASI - SESUAIKAN DENGAN NAMA FILE ANDA
# ============================================================
INPUT_FILE = 'ANALISIS PENGARUH FAKTOR GAYA HIDUP DAN KEBIASAAN BELAJAR TERHADAP KINERJA AKADEMIK SISWA SMK.csv'          # Ganti dengan nama file mentah Anda
OUTPUT_FILE = 'data_normalized.csv'

# Nama kolom di CSV (cari berdasarkan kata kunci)
col_map = {
    'jam_tidur': None,
    'screen_time': None,
    'jam_belajar': None
}

# ============================================================
# 1. BACA DATA
# ============================================================
df = pd.read_csv(INPUT_FILE)

# Deteksi kolom berdasarkan kata kunci
for col in df.columns:
    if 'jam tidur' in col.lower() or 'tidur' in col.lower():
        col_map['jam_tidur'] = col
    if 'screen' in col.lower() or 'waktu layar' in col.lower() or 'bermain hp' in col.lower():
        col_map['screen_time'] = col
    if 'jam belajar' in col.lower() or 'waktu yang kamu habiskan untuk belajar' in col.lower():
        col_map['jam_belajar'] = col

# Pastikan kolom ditemukan
for k, v in col_map.items():
    if v is None:
        raise ValueError(f"Kolom untuk {k} tidak ditemukan. Periksa nama kolom di CSV.")

COL_TIDUR = col_map['jam_tidur']
COL_SCREEN = col_map['screen_time']
COL_BELAJAR = col_map['jam_belajar']

# ============================================================
# 2. FUNGSI PEMBANTU
# ============================================================
def to_numeric(series):
    """Konversi ke numerik, abaikan karakter aneh"""
    return pd.to_numeric(series.astype(str).str.replace(',', '.').str.strip(), errors='coerce')

def is_hhmm_format(val):
    """Deteksi format HHMM (misal 2312)"""
    if pd.isna(val):
        return False
    try:
        s = str(int(val)).strip()
        if re.match(r'^\d{3,4}$', s):
            num = int(s)
            hour = num // 100
            minute = num % 100
            if 0 <= hour <= 23 and 0 <= minute <= 59:
                return True
    except:
        pass
    return False

# ============================================================
# 3. KONVERSI KE NUMERIK
# ============================================================
df[COL_TIDUR] = to_numeric(df[COL_TIDUR])
df[COL_SCREEN] = to_numeric(df[COL_SCREEN])
df[COL_BELAJAR] = to_numeric(df[COL_BELAJAR])

# ============================================================
# 4. DETEKSI DAN HAPUS ANOMALI
# ============================================================
anomali = {'hhmm': 0, '>12': {COL_TIDUR:0, COL_SCREEN:0, COL_BELAJAR:0}}

# Jam Tidur: HHMM atau >12
mask_hhmm = df[COL_TIDUR].apply(is_hhmm_format)
mask_tidur_gt12 = df[COL_TIDUR] > 12
df.loc[mask_hhmm | mask_tidur_gt12, COL_TIDUR] = np.nan
anomali['hhmm'] = mask_hhmm.sum()
anomali['>12'][COL_TIDUR] = mask_tidur_gt12.sum()

# Screen Time: >12
mask_sc_gt12 = df[COL_SCREEN] > 12
df.loc[mask_sc_gt12, COL_SCREEN] = np.nan
anomali['>12'][COL_SCREEN] = mask_sc_gt12.sum()

# Jam Belajar: >12
mask_bljr_gt12 = df[COL_BELAJAR] > 12
df.loc[mask_bljr_gt12, COL_BELAJAR] = np.nan
anomali['>12'][COL_BELAJAR] = mask_bljr_gt12.sum()

print("Anomali terdeteksi:")
print(f"  Format HHMM pada jam tidur: {anomali['hhmm']} baris")
print(f"  Jam tidur > 12: {anomali['>12'][COL_TIDUR]} baris")
print(f"  Screen time > 12: {anomali['>12'][COL_SCREEN]} baris")
print(f"  Jam belajar > 12: {anomali['>12'][COL_BELAJAR]} baris")

# ============================================================
# 5. ISI NAN DENGAN MEDIAN
# ============================================================
median_tidur = df[COL_TIDUR].median()
median_screen = df[COL_SCREEN].median()
median_belajar = df[COL_BELAJAR].median()

df[COL_TIDUR] = df[COL_TIDUR].fillna(median_tidur)
df[COL_SCREEN] = df[COL_SCREEN].fillna(median_screen)
df[COL_BELAJAR] = df[COL_BELAJAR].fillna(median_belajar)

print(f"\nMedian jam tidur: {median_tidur:.1f} jam")
print(f"Median screen time: {median_screen:.1f} jam")
print(f"Median jam belajar: {median_belajar:.1f} jam")

# ============================================================
# 6. KLIP NILAI AGAR LOGIS
# ============================================================
df[COL_TIDUR] = df[COL_TIDUR].clip(lower=0, upper=12)
df[COL_SCREEN] = df[COL_SCREEN].clip(lower=0, upper=12)
df[COL_BELAJAR] = df[COL_BELAJAR].clip(lower=0, upper=12)

# ============================================================
# 7. SIMPAN
# ============================================================
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nData tersimpan di '{OUTPUT_FILE}'")

Anomali terdeteksi:
  Format HHMM pada jam tidur: 0 baris
  Jam tidur > 12: 19 baris
  Screen time > 12: 7 baris
  Jam belajar > 12: 3 baris

Median jam tidur: 10.0 jam
Median screen time: 3.0 jam
Median jam belajar: 2.0 jam

Data tersimpan di 'data_normalized.csv'


In [6]:
import pandas as pd
import numpy as np
import re

# ============================================================
# KONFIGURASI
# ============================================================
INPUT_FILE = 'data_normalized.csv'     # Hasil dari normalisasi.py (atau raw_survey.csv jika sudah bersih)
OUTPUT_FILE = 'data_17_variabel.csv'

# ============================================================
# 1. BACA DATA
# ============================================================
df = pd.read_csv(INPUT_FILE)

# ============================================================
# 2. PEMETAAN KOLOM BERDASARKAN KATA KUNCI
# ============================================================
col_map = {}
for col in df.columns:
    if 'nisn' in col.lower():
        col_map['nisn'] = col
    elif 'nama lengkap' in col.lower() or 'nama siswa' in col.lower():
        col_map['nama'] = col
    elif 'username' in col.lower() or 'email' in col.lower():
        col_map['email'] = col
    elif 'jenis kelamin' in col.lower() or 'gender' in col.lower():
        col_map['gender'] = col
    elif 'pendidikan terakhir ayah' in col.lower():
        col_map['pend_ayah'] = col
    elif 'pendidikan terakhir ibu' in col.lower():
        col_map['pend_ibu'] = col
    elif 'pemasukan keluarga' in col.lower() or 'pendapatan' in col.lower():
        col_map['pendapatan'] = col
    elif 'pekerjaan sampingan' in col.lower():
        col_map['kerja_sampingan'] = col
    elif 'jam belajar' in col.lower() or 'waktu yang kamu habiskan untuk belajar' in col.lower():
        col_map['jam_belajar'] = col
    elif 'jam tidur' in col.lower():
        col_map['jam_tidur'] = col
    elif 'screen' in col.lower() or 'bermain hp' in col.lower():
        col_map['screen_time'] = col
    elif 'masalah aneh' in col.lower() or 'insting pertamamu' in col.lower():
        col_map['q9'] = col
    elif 'supervisor' in col.lower() or 'aplikasi atau alat brand baru' in col.lower():
        col_map['q10'] = col
    elif 'kemampuan keahlian praktik' in col.lower() or 'kompetensi' in col.lower():
        col_map['kompetensi'] = col
    elif 'konsentrasi penuh' in col.lower() or 'lokasi fisik' in col.lower():
        col_map['q11'] = col
    elif 'posisi hp' in col.lower() or 'smartphone' in col.lower():
        col_map['q12'] = col
    elif 'ujian teori besok' in col.lower():
        col_map['q13'] = col
    elif 'grup whatsapp' in col.lower() or 'mabar' in col.lower():
        col_map['q14'] = col
    elif 'titik jenuh' in col.lower() or 'pikiran realistis' in col.lower():
        col_map['q15'] = col
    elif 'fisik sudah cukup lelah' in col.lower() or 'refleks pertamamu' in col.lower():
        col_map['q16'] = col
    elif 'kritikan tajam' in col.lower() or 'omelan' in col.lower():
        col_map['q17'] = col
    elif 'teman-teman di sekitarmu' in col.lower() or 'suara batin' in col.lower():
        col_map['q18'] = col

# Verifikasi
required = ['nisn', 'nama', 'email', 'gender', 'pend_ayah', 'pend_ibu', 'pendapatan', 'kerja_sampingan',
            'jam_belajar', 'jam_tidur', 'screen_time', 'q9','q10','kompetensi','q11','q12','q13','q14','q15','q16','q17','q18']
for r in required:
    if r not in col_map:
        raise ValueError(f"Kolom {r} tidak ditemukan!")

# ============================================================
# 3. FUNGSI BANTU
# ============================================================
def normalize_text(text):
    """Membersihkan teks: lower, hapus spasi berlebih, newline, karakter khusus"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)   # spasi ganda, tab, newline → spasi tunggal
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'[“”]', '"', text)  # ganti tanda kutip miring dengan lurus
    return text.strip()

def match_choice(answer_text, choices_dict):
    """Cocokkan jawaban ke pilihan A/B/C dengan teks lengkap + substring + kata kunci"""
    if not isinstance(answer_text, str):
        return None
    text = normalize_text(answer_text)
    # 1) Cocok persis setelah normalisasi
    for choice, full_text in choices_dict.items():
        if text == normalize_text(full_text):
            return choice
    # 2) Jika teks kunci ada di dalam jawaban (substring)
    for choice, full_text in choices_dict.items():
        if normalize_text(full_text) in text:
            return choice
    # 3) Fallback kata kunci pendek (untuk Q15 dll.)
    # Q15 kata kunci spesifik
    if 'tunda-tunda terus' in text and 'guru' in text:
        return 'A'
    if 'target kebebasan finansial' in text or 'tabungan 150 juta' in text or 'masa depan' in text:
        return 'B'
    if 'sensasi rasa lega' in text or 'kepuasan batin' in text or 'berhasil memecahkan' in text:
        return 'C'
    return None

def score_question(choice, score_map):
    if choice in score_map:
        return score_map[choice]
    return np.nan

def clean_num(val):
    """Konversi ke float, tangani spasi/koma"""
    if isinstance(val, str):
        val = val.replace(',', '.').strip()
        val = re.sub(r'\s+', '', val)
        try:
            return float(val)
        except:
            return np.nan
    return float(val) if pd.notna(val) else np.nan

# ============================================================
# 4. DEFINISI TEKS PILIHAN & SKOR
# ============================================================
q9_choices = {
    'A': 'Segera melepaskan alat tersebut dan melaporkannya ke pembimbing PKL atau teknisi senior.',
    'B': 'Memperhatikan pesan error di layar, atau mencari sumber bunyi alat secara spesifik, lalu mencari gejala tersebut di Google/YouTube/Forum hardware.',
    'C': 'Langsung melakukan cara cepat: restart PC, cabut-pasang kabel, atau menekan tombol reset pada alat.'
}
q9_scores = {'A':30, 'B':50, 'C':10}

q10_choices = {
    'A': 'Menerima tugas tersebut, lalu diam-diam mencari tutorial dasar di YouTube atau membaca manual book-nya untuk mencoba memahami dasar kerjanya secara otodidak, sebelum mulai mengerjakan tugas.',
    'B': 'Meminta izin untuk tetap menggunakan aplikasi atau alat yang sudah diajarkan di sekolah saja.',
    'C': 'Menerima tugas tersebut, tetapi meminta pembimbing untuk mencontohkan langkah demi langkah penggunaannya dari awal sampai akhir, karena kamu takut melakukan kesalahan yang merugikan perusahaan.'
}
q10_scores = {'A':50, 'B':10, 'C':30}

q11_choices = {
    'A': 'Kamu lebih suka mengerjakan di ruang keluarga, ruang TV, atau tempat nongkrong yang ada sedikit keramaian.',
    'B': 'Kamu mencari meja khusus atau sudut ruangan yang kosong, lalu menyingkirkan barang-barang yang tidak berhubungan dengan tugas tersebut.',
    'C': 'Kamu memilih mengerjakannya di atas kasur sambil rebahan, atau duduk bersandar di sofa yang empuk.'
}
q11_scores = {'A':30, 'B':50, 'C':10}

q12_choices = {
    'A': 'Menaruh HP tepat di sebelah laptop atau mouse dengan layar menghadap ke atas.',
    'B': 'Memasukkan HP ke dalam laci, ke dalam tas, atau menaruhnya di seberang ruangan yang mengharuskanmu berdiri untuk mengambilnya.',
    'C': 'Menaruh HP di atas meja kerjamu, tetapi posisinya dibalik (layar menghadap bawah) dan di-set ke mode getar/senyap.'
}
q12_scores = {'A':10, 'B':50, 'C':30}

q13_choices = {
    'A': 'Mengeksekusi ide praktik tersebut sekarang juga mumpung sedang ada inspirasi dan mood yang bagus.',
    'B': 'Memaksakan diri belajar teori dari sekarang dan memblokir pikiran tentang ide praktik tersebut.',
    'C': 'Langsung mengambil HP atau buku catatan, menuliskan ide praktik tersebut sedetail mungkin agar tidak lupa, lalu menyimpannya. Setelah itu sisa waktu hari ini digunakan 100% untuk fokus belajar ujian teori.'
}
q13_scores = {'A':10, 'B':30, 'C':50}

q14_choices = {
    'A': 'Membaca obrolan itu sekilas saja sambil tetap mengetik laporan (melakukan multitasking).',
    'B': 'Ikut membalas obrolan sebentar karena merasa kebetulan butuh refreshing / istirahat sejenak.',
    'C': 'Kamu tidak tahu grup itu ramai karena sejak awal mulai menyusun laporan, HP sudah kamu jauhkan dari pandangan mata atau diatur ke mode Do Not Disturb (Jangan Ganggu).'
}
q14_scores = {'A':10, 'B':30, 'C':50}

# Q15 - Motivasi Akademik (dengan kata kunci yang lebih longgar)
q15_scores = {'A':20, 'B':60, 'C':100}

q16_choices = {
    'A': 'Kamu mencari alasan untuk keluar sebentar (misalnya ke toilet agak lama) atau pura-pura sibuk merapikan alat di meja belakang.',
    'B': 'Kamu langsung menyiapkan alat dan bahan, membagi langkah kerjanya di kepala, dan mulai mengerjakan pelan-pelan secara urut.',
    'C': 'Kamu panik dan merakit/mengerjakan secara acak asalkan terlihat ada progres.'
}
q16_scores = {'A':35, 'B':10, 'C':20}

q17_choices = {
    'A': 'Mengiyakan dan mengangguk saja sambil menunduk agar omelannya cepat selesai.',
    'B': 'Menerima kritikan tersebut sebagai simulasi tekanan kerja nanti di industri.',
    'C': 'Merasa omelan itu berlebihan atau terlalu idealis.'
}
q17_scores = {'A':20, 'B':10, 'C':35}

q18_choices = {
    'A': 'Menghibur diri dengan asumsi bahwa alat bantu, kabel, atau komputer yang mereka pakai kebetulan kondisinya lebih bagus dan normal, sementara kamu sial mendapatkan alat yang rewel atau susah diatur.',
    'B': 'Merasa bahwa kamu memang "salah jurusan" atau tidak punya "bakat teknik".',
    'C': 'Menjadikan pekerjaan mereka sebagai referensi (contek ilmu).'
}
q18_scores = {'A':15, 'B':30, 'C':5}

# ============================================================
# 5. PROSES SETIAP BARIS
# ============================================================
output = []
for idx, row in df.iterrows():
    nisn = str(row[col_map['nisn']]).strip()
    nama = str(row[col_map['nama']]).strip()
    email = str(row[col_map['email']]).strip()
    gender_raw = str(row[col_map['gender']]).strip()
    gender = 'Laki-laki' if 'laki' in gender_raw.lower() else 'Perempuan'

    jam_belajar = clean_num(row[col_map['jam_belajar']])
    jam_tidur = clean_num(row[col_map['jam_tidur']])
    screen_time = clean_num(row[col_map['screen_time']])

    pend_ayah = str(row[col_map['pend_ayah']]).strip()
    pend_ibu = str(row[col_map['pend_ibu']]).strip()
    def map_pendidikan(text):
        t = text.lower()
        if 'sarjana' in t or 's1' in t or 'master' in t or 'doctor' in t:
            return 'Sarjana'
        if 'diploma' in t:
            return 'Diploma'
        if 'sma' in t:
            return 'SMA/SMK'
        if 'smp' in t:
            return 'SMP'
        if 'sd' in t:
            return 'SD'
        return text
    p_ayah = map_pendidikan(pend_ayah)
    p_ibu = map_pendidikan(pend_ibu)
    order = {'SD':1, 'SMP':2, 'SMA/SMK':3, 'Diploma':4, 'Sarjana':5}
    pendidikan_tertinggi = p_ayah if order.get(p_ayah,0) >= order.get(p_ibu,0) else p_ibu

    pend_raw = str(row[col_map['pendapatan']]).strip().lower()
    if 'kurang dari' in pend_raw or '<' in pend_raw:
        pendapatan = '< 2 Juta'
    elif '2.000.000' in pend_raw or '2.000.000 sampai' in pend_raw:
        pendapatan = '2 - 5 Juta'
    elif '5.000.000' in pend_raw or '5.000.000 sampai' in pend_raw:
        pendapatan = '5 - 10 Juta'
    elif '> 10' in pend_raw or 'lebih' in pend_raw:
        pendapatan = '> 10 Juta'
    else:
        pendapatan = pend_raw

    kerja_raw = str(row[col_map['kerja_sampingan']]).strip().lower()
    if 'ya' in kerja_raw:
        kerja_sampingan = 'Ya'
    elif 'tidak' in kerja_raw:
        kerja_sampingan = 'Tidak'
    else:
        kerja_sampingan = kerja_raw

    def get_choice(col_key, choices_dict):
        ans = row[col_map[col_key]]
        if pd.isna(ans) or str(ans).strip() == '':
            return None
        return match_choice(str(ans), choices_dict)

    # Q9, Q10
    q9_c = get_choice('q9', q9_choices)
    q10_c = get_choice('q10', q10_choices)
    q9_score = score_question(q9_c, q9_scores)
    q10_score = score_question(q10_c, q10_scores)
    industry_sum = q9_score + q10_score
    industry_readiness = 'Belum Siap' if (pd.isna(industry_sum) or industry_sum <= 50) else 'Siap'

    # Q11, Q12
    q11_c = get_choice('q11', q11_choices)
    q12_c = get_choice('q12', q12_choices)
    q11_score = score_question(q11_c, q11_scores)
    q12_score = score_question(q12_c, q12_scores)
    env_sum = q11_score + q12_score
    if pd.isna(env_sum):
        study_environment = 'Cukup Kondusif'
    elif env_sum <= 40:
        study_environment = 'Kurang Kondusif'
    elif env_sum <= 70:
        study_environment = 'Cukup Kondusif'
    else:
        study_environment = 'Kondusif'

    # Time management
    q13_c = get_choice('q13', q13_choices)
    q14_c = get_choice('q14', q14_choices)
    q13_score = score_question(q13_c, q13_scores)
    q14_score = score_question(q14_c, q14_scores)
    skor_time_management = q13_score + q14_score

    # Motivasi (Q15) – gunakan fungsi terpisah yang sudah ada di dalam match_choice fallback
    q15_raw = str(row[col_map['q15']]) if pd.notna(row[col_map['q15']]) else ''
    q15_c = match_choice(q15_raw, {})  # tidak perlu choices_dict, karena fallback menangani
    if q15_c is None:
        # coba paksa deteksi kata kunci
        q15_c = match_choice(q15_raw, {})   # sudah ada fallback di match_choice
    motivasi_akademik = score_question(q15_c, q15_scores)

    # Stress Q16-18
    q16_c = get_choice('q16', q16_choices)
    q17_c = get_choice('q17', q17_choices)
    q18_c = get_choice('q18', q18_choices)
    q16_score = score_question(q16_c, q16_scores)
    q17_score = score_question(q17_c, q17_scores)
    q18_score = score_question(q18_c, q18_scores)
    stress_sum = q16_score + q17_score + q18_score
    if pd.isna(stress_sum):
        stress_level = 'Sedang'
    elif stress_sum <= 45:
        stress_level = 'Rendah'
    elif stress_sum <= 70:
        stress_level = 'Sedang'
    else:
        stress_level = 'Berat'

    # Kompetensi skill
    komp_raw = str(row[col_map['kompetensi']]).strip().lower()
    if 'sangat menguasai' in komp_raw:
        kompetensi = 'Tinggi'
    elif 'masih sering bingung' in komp_raw:
        kompetensi = 'Rendah'
    else:
        kompetensi = 'Menengah'

    presentase_kehadiran = np.nan
    nilai_rata_rata_raport = np.nan
    kehadiran_pelatihan_industry = np.nan
    exam_score = np.nan

    output.append({
        'nisn': nisn,
        'nama_siswa': nama,
        'email': email,
        'jam_belajar_per_hari': jam_belajar,
        'presentase_kehadiran': presentase_kehadiran,
        'nilai_rata_rata_raport': nilai_rata_rata_raport,
        'skor_time_management': skor_time_management,
        'jam_tidur': jam_tidur,
        'screen_time': screen_time,
        'kehadiran_pelatihan_industry': kehadiran_pelatihan_industry,
        'motivasi_akademik': motivasi_akademik,
        'exam_score': exam_score,
        'gender': gender,
        'rata_rata_pemasukan_keluarga': pendapatan,
        'pendidikan_terakhir_orang_tua': pendidikan_tertinggi,
        'kerja_sampingan': kerja_sampingan,
        'study_environment': study_environment,
        'kompetensi_skill_level': kompetensi,
        'industry_readiness': industry_readiness,
        'stress_level': stress_level
    })

final_df = pd.DataFrame(output)
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Konversi selesai. File disimpan sebagai '{OUTPUT_FILE}'")
print(f"Total baris: {len(final_df)}")
print(final_df.head())

Konversi selesai. File disimpan sebagai 'data_17_variabel.csv'
Total baris: 197
         nisn           nama_siswa                      email  \
0  3105858845    M. Farid alfarisi      faridmdtmdt@gmail.com   
1  0095714311           Almuntazar         tadarspn@gmail.com   
2  0109189747               Furkan  farelfarel97936@gmail.com   
3  0082559539   Alisa Putri Ananda  alsptriananda51@gmail.com   
4   008619209  Asmaul Husna Fitria     almaulusna58@gmail.com   

   jam_belajar_per_hari  presentase_kehadiran  nilai_rata_rata_raport  \
0                   1.0                   NaN                     NaN   
1                   8.0                   NaN                     NaN   
2                   1.0                   NaN                     NaN   
3                   7.0                   NaN                     NaN   
4                   2.0                   NaN                     NaN   

   skor_time_management  jam_tidur  screen_time  kehadiran_pelatihan_industry  \
0        